# v4 Transformer Chat — Load Any Checkpoint

Load a specific epoch/step checkpoint and chat with that version of the model.

1. Run all setup cells (1-3)
2. Set `CHECKPOINT_PATH` to the `.pt` file you want to chat with
3. Run the load cell
4. Run the chat cell — type messages in Gradio

In [ ]:
# Cell 1: Imports
import os, re, math, copy, time, torch, warnings
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
from os.path import exists
import spacy

warnings.filterwarnings("ignore")
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

In [ ]:
# Cell 2: Which mode was used to train? (must match the checkpoint architecture)
# Options: "local", "local_v4", "full", "kaggle_v4"
TRAIN_MODE = "local_v4"

# Path to the checkpoint you want to chat with
# Examples:
#   "v4_local_mixchat_epoch_0.pt"   — after epoch 0
#   "v4_local_mixchat_epoch_3.pt"   — after epoch 3
#   "v4_local_mixchat_last.pt"      — latest saved state
#   "v4_local_mixchat_final.pt"     — after all epochs done
#   "v4_kaggle_mixchat_epoch_2.pt"  — kaggle epoch 2
CHECKPOINT_PATH = "v4_local_mixchat_final.pt"

print(f"Mode: {TRAIN_MODE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Exists: {exists(CHECKPOINT_PATH)}")

In [ ]:
# Cell 3: Model architecture (same as transformer.ipynb — needed to rebuild the model)

def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size
    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)
    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)

class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)
    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)

class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator
    def forward(self, src, tgt, src_mask, tgt_mask):
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)
    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)
    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super().__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)
    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        del query, key, value
        return self.linears[-1](x)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))

class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model
    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)
    def forward(self, x):
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)

def subsequent_mask(size):
    attn_shape = (1, size, size)
    return torch.triu(torch.ones(attn_shape), diagonal=1).type(torch.uint8) == 0

def make_model(src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1):
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab),
    )
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model

def greedy_decode(model, src, src_mask, max_len, start_symbol):
    memory = model.encode(src, src_mask)
    ys = torch.zeros(1, 1).fill_(start_symbol).type_as(src.data)
    for i in range(max_len - 1):
        out = model.decode(memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src.data))
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, torch.zeros(1, 1).type_as(src.data).fill_(next_word)], dim=1)
    return ys

print("Architecture defined.")

In [ ]:
# Cell 4: Config lookup (must match what was used during training)
CONFIGS = {
    "local": {"N": 2, "d_model": 512, "d_ff": 2048, "h": 8, "dropout": 0.1, "max_padding": 48,
              "vocab_path": "vocab_local_all.json"},
    "local_v4": {"N": 4, "d_model": 512, "d_ff": 2048, "h": 8, "dropout": 0.1, "max_padding": 64,
                 "vocab_path": "v4_vocab_local_mixchat.json"},
    "full": {"N": 6, "d_model": 512, "d_ff": 2048, "h": 8, "dropout": 0.1, "max_padding": 72,
             "vocab_path": "vocab.json"},
    "kaggle_v4": {"N": 6, "d_model": 768, "d_ff": 3072, "h": 12, "dropout": 0.1, "max_padding": 96,
                  "vocab_path": "v4_vocab_kaggle_mixchat.json"},
}

cfg = CONFIGS[TRAIN_MODE]
print(f"Config for {TRAIN_MODE}:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 5: Load vocabulary
from tokenizers import Tokenizer

vocab_path = cfg["vocab_path"]
if not exists(vocab_path):
    raise FileNotFoundError(f"Vocab file not found: {vocab_path}. Train the model first.")

vocab = Tokenizer.from_file(vocab_path)
print(f"Vocabulary loaded: {vocab.get_vocab_size()} tokens from {vocab_path}")

spacy_en = spacy.load("en_core_web_sm")
print("Tokenizer ready.")

In [ ]:
# Cell 6: Load the checkpoint and build model
if not exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

vocab_size = vocab.get_vocab_size()
model = make_model(
    vocab_size, vocab_size,
    N=cfg["N"], d_model=cfg["d_model"], d_ff=cfg["d_ff"],
    h=cfg["h"], dropout=cfg["dropout"],
)

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
if isinstance(ckpt, dict) and "model" in ckpt:
    model.load_state_dict(ckpt["model"])
    epoch = ckpt.get("epoch", "?")
    step = ckpt.get("step", "?")
    val_loss = ckpt.get("val_loss", "?")
    print(f"Loaded checkpoint: epoch={epoch}, step={step}, val_loss={val_loss}")
else:
    model.load_state_dict(ckpt)
    print("Loaded raw state_dict (no metadata)")

model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model ready: {total_params:,} parameters")

In [ ]:
# Cell 7: List all available checkpoints you can chat with
import glob

print("Available checkpoints in current directory:")
print("-" * 50)
for pt_file in sorted(glob.glob("*.pt")):
    size_mb = os.path.getsize(pt_file) / (1024 * 1024)
    print(f"  {pt_file:45s}  ({size_mb:.1f} MB)")

print()
print(f"Currently loaded: {CHECKPOINT_PATH}")
print("To switch: change CHECKPOINT_PATH in Cell 2 and re-run cells 6-8")

In [ ]:
# Cell 8: Gradio Chat UI
import gradio as gr

def tokenize(text, tokenizer):
    return [tok.text for tok in tokenizer.tokenizer(text)]

max_padding = cfg["max_padding"]
pad_idx = vocab.token_to_id("<blank>")
start_symbol = vocab.token_to_id("<s>")
eos_id = vocab.token_to_id("</s>")

def chat_reply(user_text, history):
    model.eval()
    src_tokens = tokenize(user_text, spacy_en)
    src_ids = vocab.encode(" ".join(src_tokens)).ids[: max(1, max_padding - 2)]
    src_tensor = torch.tensor(
        [vocab.token_to_id("<s>")] + src_ids + [eos_id], dtype=torch.long
    ).unsqueeze(0)
    if src_tensor.size(1) < max_padding:
        src_tensor = torch.nn.functional.pad(
            src_tensor, (0, max_padding - src_tensor.size(1)), value=pad_idx
        )
    src_mask = (src_tensor != pad_idx).unsqueeze(-2)
    out = greedy_decode(model, src_tensor, src_mask, max_padding, start_symbol)
    reply_tokens = [vocab.id_to_token(x) for x in out[0].tolist() if x != pad_idx]
    reply = " ".join(reply_tokens).replace("</s>", "").replace("<s>", "").strip()
    if not reply:
        reply = "(empty reply)"
    history = history or []
    history.append({"role": "user", "content": user_text})
    history.append({"role": "assistant", "content": reply})
    return history, history

with gr.Blocks(title=f"v4 Chat — {CHECKPOINT_PATH}") as demo:
    gr.Markdown(f"## Chatting with: `{CHECKPOINT_PATH}` ({TRAIN_MODE} config)")
    chatbot = gr.Chatbot()
    state = gr.State([])
    txt = gr.Textbox(placeholder="Type a message and press Enter...")
    txt.submit(chat_reply, [txt, state], [chatbot, state])
    txt.submit(lambda: "", None, txt)

demo.launch(share=False)